# C01 · Golf putting: when physics beats curve fitting

| | |
|---|---|
| **Difficulty** | ★★☆☆☆ |
| **Time** | 2-3 hours |
| **Data** | Berry (1996) pro putting data; Broadie's modern PGA data |
| **Skills** | Binomial likelihoods · building a likelihood from a mechanistic model · prior predictive checks · LOO model comparison · diagnosing a sampler that refuses to mix · model expansion |

## The brief

A golf analytics company wants a model for the probability that a professional sinks a putt
as a function of distance. It will feed a "strokes gained" product, so it has to
**extrapolate sensibly** and its parameters should **mean something** to a coach.

You get two datasets: a small classic one (19 distances, ~5,600 putts) and a modern one
with 30x more putts. A model that looks great on the first will break on the second.
Your job is to find out why and fix it.

## How this notebook works

- Each task states **what to deliver**, not how. Write your code in the `YOUR CODE HERE` cells.
- Stuck? `h.hint("task2")` reveals hints one level at a time: *nudge → approach → code skeleton*.
  Try to get by on nudges.
- `h.check("task2", sigma_angle_deg=...)` compares your numbers with the reference solution.
- A full worked solution lives in `notebooks/solutions/`. Open it only when you are done (or truly stuck).

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
from scipy import stats

from pymc_challenges import Hints, data

RANDOM_SEED = 1996
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-variat")

h = Hints("C01")
h.tasks()

## The data

`x` is the distance to the hole in feet, `n` the number of putts attempted from that
distance and `y` how many went in.

In [ ]:
data.describe("golf")
golf = data.load("golf")
golf.head()

Two physical constants you will need later (both in **feet**, like `x`):

In [ ]:
BALL_RADIUS = (1.68 / 2) / 12
HOLE_RADIUS = (4.25 / 2) / 12

## Task 0 · Look at the data

Plot the empirical success rate against distance **with an uncertainty interval for each
point**. Which distances are measured precisely, and which are not?

In [ ]:
# YOUR CODE HERE

## Task 1 · The default: logistic regression

Fit the model every analyst reaches for first:

$$y_i \sim \text{Binomial}(n_i, p_i), \qquad \text{logit}(p_i) = a + b\,x_i$$

**Deliver**
1. Priors you can defend. *Show* with a prior predictive plot that they imply plausible
   curves (no 50% success at 0 feet, no cliff edges).
2. A fitted model with clean diagnostics (`r_hat`, ESS, divergences).
3. A plot of the posterior success curve with a credible band on top of the data.
   Where does the fit fail, and in which direction?

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task1")
# h.check("task1", a=..., b=...)   # posterior means

## Task 2 · Think like a golfer

A putt goes in if the ball's centre passes over the hole. Suppose the golfer aims at the
centre but the launch **angle** is off by a random error $\theta \sim \text{Normal}(0, \sigma)$.

**Deliver**
1. Derive $p(x)$ from the geometry (constants `BALL_RADIUS`, `HOLE_RADIUS` are above) and
   write it down in a markdown cell. It has **one** free parameter.
2. Implement it in PyMC and fit it. Report $\sigma$ in **degrees**.
3. Overlay its posterior curve on the logistic fit. One parameter versus two - which wins?

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task2")
# h.check("task2", sigma_angle_deg=...)   # posterior mean, in degrees

## Task 3 · Make the comparison quantitative

Compare the two models with PSIS-LOO.

**Deliver** the `az.compare` table and two sentences: what does it say, and how far do you
trust it here? (Look at the warnings - there are only 19 "observations".)

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task3")

## Task 4 · New data breaks the model

Mark Broadie collected far more putts, out to 75 feet.

**Deliver**
1. Fit your angle model to `golf_new`, unchanged. Plot fit against data, and plot the
   **residuals** (observed rate − posterior mean). What is the pattern?
2. A one-paragraph hypothesis: what real-world mechanism is the model missing?

In [ ]:
data.describe("golf_new")
golf_new = data.load("golf_new")
golf_new.tail()

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task4")

## Task 5 · Add distance control (and survive the sampler)

Domain knowledge from a putting coach:

- Golfers aim to hit the ball about **1 foot past** the hole.
- The putt can drop if the ball would have travelled between the hole and **3 feet past** it.
- The error in how far the ball travels is **proportional to the intended distance**.

**Deliver**
1. Turn that into $p_\text{distance}(x)$ with one new parameter, and combine it with the
   angle component (assume independence).
2. Fit it with a Binomial likelihood. **Check the diagnostics carefully.** Something will be
   wrong - explain *why* this posterior is hard, in terms of the data.
3. Fix the model (not the sampler settings!) and refit. Report all parameters and show the
   final fit and residuals.

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task5")
# h.check("task5", sigma_angle_deg=..., sigma_distance=...)   # posterior means, final model

## Task 6 · Answer the business question

The product team asks: **"From what distance is a tour pro a coin flip?"**

**Deliver** the posterior distribution of $x_{50}$, the distance at which $p(x) = 0.5$,
with a point estimate and a 94% interval. Then the follow-up: a coaching intervention
claims to cut distance error $\sigma_d$ by a fifth. How many extra putts per 100 would
that sink from 30 feet?

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task6")
# h.check("task6", x50=...)   # posterior mean, in feet

## Going further

- The final residuals still wiggle systematically between 3 and 12 feet. Propose and test a mechanism.
- Refit the final model with the exact Binomial likelihood plus a *Beta-Binomial* or a
  logit-normal random effect per distance instead of the Normal approximation. Do the
  conclusions change?
- Fit both datasets jointly with shared physics but dataset-specific skill parameters. Did
  pros improve between the 1990s and the 2010s?

In [ ]:
h.progress()